In [1]:
print("ok")

ok


In [2]:
%pwd

'd:\\Build-a-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS\\research'

In [3]:
import os

os.chdir("../")

In [4]:
%pwd

'd:\\Build-a-Complete-Medical-Chatbot-with-LLMs-LangChain-Pinecone-Flask-AWS'

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [10]:
from typing import List
from langchain.schema import Document

In [ ]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
#split documents into chunks

def split_documents(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [ ]:
text_chunk = split_documents(minimal_docs)
print(f"text chunk:{len(text_chunk)}")
text_chunk

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download the embeddings model from HuggingFace and return the embeddings object.
    """
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings = download_embeddings()

In [ ]:
embeddings

In [ ]:
vector = embeddings.embed_query("What is the capital of France?")

vector

In [ ]:
print("length of vector:", len(vector))

In [12]:
from dotenv import load_dotenv

import os

load_dotenv()

True

In [13]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [14]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [15]:
pc

In [20]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws" , region="us-east-1" )
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
#load existing Index

docsearch = PineconeVectorStore.from_existing_index(embedding=embeddings, index_name=index_name)


In [ ]:
#Add more data to the existing index

dummy_text = Document(
    page_content="This is a dummy document for testing purposes.",
    metadata={"source": "dummy_source"}
)

In [ ]:
docsearch.add_documents(documents=[dummy_text])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("What is the capital of France?")
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
     "You are a medical chatbot that provides information based on the documents provided. "
     "Use the documents to answer the user's questions. If the answer is not in the documents, respond with 'I don't know."
     "use the three sentence if u dont know the answer to the question and provide a relevant answer based on the documents provided."
     "\n\n"
     "{context}"
   )

prompt = ChatPromptTemplate.from_messsages(
  [
    ("system", system_prompt),
    ("human", "{input}"),
  ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain= create_retrieval_chain(retriever, question_answer_chain)


In [ ]:
response = rag_chain.invoke({"input ":"What is the capital of France?"})
print(response[answer])